In [3]:
import sys, xgboost
print(sys.executable)
print(sys.version)
print("xgboost:", xgboost.__version__)


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score, fbeta_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load Data
df_eng = pd.read_csv('../data/processed/insurance_claims_engineered_final.csv')
df_pre = pd.read_csv('../data/processed/insurance_claims_preprocessed_no_hobbies.csv')
df_trees = pd.read_csv('../data/processed/preprocesed_for_trees.csv')

# 2. Prepare Data & Ensure Alignment
SEED = 42
y = df_eng['target'] 

# Split Engineered Data (For LogReg)
X_eng = df_eng.drop(columns=['target'])
X_train_eng, X_test_eng, y_train, y_test = train_test_split(X_eng, y, test_size=0.20, stratify=y, random_state=SEED)

# Split Preprocessed Data (For Trees - Original)
X_pre = df_pre.drop(columns=['target'])
X_train_pre, X_test_pre, _, _ = train_test_split(X_pre, y, test_size=0.20, stratify=y, random_state=SEED)

# Split Trees Data (Full - with Hobbies)
X_trees = df_trees.drop(columns=['target'])
X_train_trees, X_test_trees, _, _ = train_test_split(X_trees, y, test_size=0.20, stratify=y, random_state=SEED)

# Split Trees Data (No Hobbies)
if 'insured_hobbies' in df_trees.columns:
    X_trees_no_hobby = df_trees.drop(columns=['target', 'insured_hobbies'])
else:
    X_trees_no_hobby = df_trees.drop(columns=['target'])

X_train_trees_nh, X_test_trees, _, _ = train_test_split(X_trees_no_hobby, y, test_size=0.20, stratify=y, random_state=SEED)

# 3. Define Preprocessing
def get_column_types(df):
    numeric = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return numeric, categorical

def create_preprocessor(X):
    num, cat = get_column_types(X)
    return ColumnTransformer(transformers=[
        ('num', 'passthrough', num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
    ])

# LogReg Preprocessor (Standard Scaler)
num_eng, cat_eng = get_column_types(X_eng)
preprocessor_eng = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_eng),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_eng)
])

preprocessor_pre = create_preprocessor(X_pre)
preprocessor_trees = create_preprocessor(X_trees)
preprocessor_trees_nh = create_preprocessor(X_trees_no_hobby)

# 4. Define Models
def get_et_model():
    return ExtraTreesClassifier(
        class_weight='balanced',
        n_estimators=300,
        min_samples_split=5, 
        min_samples_leaf=4,
        max_depth=10,
        max_features='log2',
        random_state=SEED
    )

model_logreg = Pipeline([
    ('prep', preprocessor_eng),
    ('clf', LogisticRegression(class_weight='balanced', solver='lbfgs', C=0.1, max_iter=5000, random_state=SEED))
])

model_et = Pipeline([('prep', preprocessor_pre), ('clf', get_et_model())])
model_et_trees = Pipeline([('prep', preprocessor_trees), ('clf', get_et_model())])
model_et_trees = Pipeline([('prep', preprocessor_trees_nh), ('clf', get_et_model())])

# 5. Train all Models
print("Training Models...")
model_logreg.fit(X_train_eng, y_train)
model_et.fit(X_train_pre, y_train)
model_et_trees.fit(X_train_trees, y_train)
model_et_trees.fit(X_train_trees_nh, y_train)

# 6. Generate Reports (Individual)
models = {
    "LogReg (Engineered)": (model_logreg, X_test_eng),
    "ExtraTrees (Base No Hobbies)": (model_et, X_test_pre),
    "ExtraTrees (Trees w/ Hobbies)": (model_et_trees, X_test_trees),
    "ExtraTrees (Trees w/o Hobbies)": (model_et_trees, X_test_trees)
}

for name, (model, X_test_data) in models.items():
    print(f"\n=== {name} ===")
    preds = model.predict(X_test_data)
    print(classification_report(y_test, preds))
    f2 = fbeta_score(y_test, preds, beta=2)
    print(f"F2 Score: {f2:.4f}")

# 7. Ensemble (Soft Voting) - FINAL CONFIGURATION
# Component 1: LogReg (Engineered)
# Component 2: ExtraTrees (Trees Dataset WITH Hobbies) <-- Updated as per request

print("\n=== FINAL ENSEMBLE (LogReg + ExtraTrees With Hobbies) ===")
probs_linear = model_logreg.predict_proba(X_test_eng)[:, 1]
probs_tree   = model_et_trees.predict_proba(X_test_trees)[:, 1]

# Average probabilities
probs_ensemble = (0.5 * probs_linear) + (0.5 * probs_tree)

# Optimize Threshold
best_th = 0.5
best_f1 = 0
best_f2 = 0

for th in np.arange(0.3, 0.7, 0.05):
    preds = (probs_ensemble >= th).astype(int)
    f1 = f1_score(y_test, preds)
    f2 = fbeta_score(y_test, preds, beta=2)
    rec = recall_score(y_test, preds)
    print(f"Threshold {th:.2f} -> Recall: {rec:.3f}, F1: {f1:.3f}, F2: {f2:.3f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_th = th
        best_f2 = f2

final_preds = (probs_ensemble >= best_th).astype(int)
final_f2 = fbeta_score(y_test, final_preds, beta=2)

print(f"\n=== ENSEMBLE PERFORMANCE (Threshold {best_th:.2f}) ===")
print(classification_report(y_test, final_preds))
print(f"Final F2 Score: {final_f2:.4f}")

In [2]:
# Version 3: Using cleaned data additional engineering 

# ============================================================
# step_01 LOAD DATA (CLEAN VERSION)
# ============================================================
# Why:
# - Load the cleaned version of the preprocessed dataset

from pathlib import Path
import pandas as pd
import os, random, numpy as np, pandas as pd
SEED=42; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"]=str(SEED)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, classification_report
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

# 🔁 Explicitly use the cleaned version of the CSV
DATA = PROC / "insurance_claims_engineered_final.csv"
assert DATA.exists(), f"File not found: {DATA}"

print("Using cleaned dataset:", DATA.name)

df = pd.read_csv(DATA)
print(df.shape); df.head(2)


In [3]:
df.columns


In [5]:
drop_cols = [
    "months_as_customer",
    "days_since_bind",
    "policy_annual_premium",
    "total_claim_amount",
    "claim_amount_risk_band",
    "policy_annual_premium_bin",
    "days_since_bind_bin",
    "auto_make",
    "auto_model"
]


In [4]:
# ============================================================
# step_02 TARGET PRESENCE & LABEL SANITY
# ============================================================
# Why:
# - Guardrail: ensure label exists and is binary.
# - Fail early if the dataset is not what we expect.

assert "target" in df.columns, f"Expected 'target' in columns, got: {df.columns.tolist()}"
unique_y = set(df["target"].unique())
assert unique_y <= {0,1}, f"Labels must be binary 0/1, got: {unique_y}"
print("Labels OK →", unique_y)


In [6]:
# ============================================================
# step_03 BUILD X / y
# ============================================================
# Why:
# - Separate features from label before ANY preprocessing.
# - This blocks label leakage later.

X = df.drop(columns=["target"])
y = df["target"]
print("X:", X.shape, "| y:", y.shape)


In [7]:
# ============================================================
# step_04 STRATIFIED TRAIN/TEST SPLIT
# ============================================================
# Why:
# - Preserve class proportions (fraud vs non-fraud) in both splits.
# - Fix seed for reproducibility.

from sklearn.model_selection import train_test_split
SEED = 42
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("Train:", X_tr.shape, " Test:", X_te.shape)
print("Pos rate → train:", y_tr.mean().round(4), "| test:", y_te.mean().round(4))


In [8]:
# ============================================================
# step_05 SAFETY CHECKS — NO LEAKAGE, RIGHT TASK
# ============================================================
# Why:
# - Now that splits exist, verify features don't contain the label.
# - Confirm labels are binary in both splits.

assert 'target' not in X_tr.columns, "Target should not be in X_tr."
assert 'target' not in X_te.columns, "Target should not be in X_te."
assert set(y_tr.unique()) <= {0,1} and set(y_te.unique()) <= {0,1}, "y must be binary 0/1"
# Silence = all good.


In [9]:
# ============================================================
# step_06 COLUMN GROUPING — NUMERIC vs CATEGORICAL (from TRAIN)
# ============================================================
# Why:
# - Numeric will be scaled; categorical will be one-hot encoded.
# - Define groups from TRAIN ONLY to avoid peeking at test.

import numpy as np
num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_tr.select_dtypes(include=['object','category']).columns.tolist()
print(f"Detected → {len(num_cols)} numeric, {len(cat_cols)} categorical")


In [10]:
# ============================================================
# step_06 COLUMN GROUPING — MANUAL + SAFE CLASSIFICATION
# ============================================================

# 1. Binary (0/1) — passthrough
binary_features = [
    "collision_type_missing",
    "property_damage_missing",
    "police_report_available_missing",
    "authorities_contacted_missing",
    "incident_weekend",
    "is_holiday",
    "holiday_window_2d",
    "is_high_risk_premium",
    "incident_severity_is_major",
    "fraud_time_window_flag"
]

# 2. Categorical — OneHotEncode
categorical_features = [
    "policy_state",
    "insured_sex",
    "insured_education_level",
    "insured_occupation",
    "insured_relationship",
    "incident_type",
    "collision_type",
    "authorities_contacted",
    "property_damage",
    "police_report_available",
    "incident_state",
    "street_type",
    "vehicle_age_bucket",
    "vehicle_tier",
    "body_type",
    "hour_bin_4",
    "days_since_bind_bucket_final",
    "customer_tenure_bucket",
    "umbrella_limit_bin",
    "incident_month",
    "incident_dow",
    
    # engineered risk band (ordinal-coded, but should be categorical)
    "claim_amount_risk_band_encoded"
]

# 3. Numeric — StandardScaled
numeric_features = [
    "age",
    "policy_deductable",
    "umbrella_limit",
    "capital-gains",
    "capital-loss",
    "number_of_vehicles_involved",
    "bodily_injuries",
    "witnesses",
    "injury_share",
    "property_share",
    "policy_bind_year",
    "policy_bind_month",
    "policy_bind_quarter",
    "hour_sin",
    "hour_cos",
    "vehicle_age",
    "csl_per_person",
    "csl_per_incident"
]

print("Binary:", len(binary_features))
print("Categorical:", len(categorical_features))
print("Numeric:", len(numeric_features))


In [11]:
# ============================================================
# step_07 PREPROCESSOR — FINAL VERSION
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("bin", "passthrough", binary_features),  # raw 0/1 flags
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Preprocessor ready!")
